In [9]:
DURATION = 200

In [10]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Параметры системы
v = 0.5
u2 = 0.6
u1_fixed = 0.97
n_iter = 100
burn_in = 20

# Сетка начальных точек для фазового портрета
x0_values = np.linspace(0.1, 0.95, 5)
y0_values = np.linspace(0.1, 0.95, 5)

# Параметры альфа для анимации
tilda_values = np.linspace(0, 280, 50)  # количество кадров

# -----------------------------
# Кэшируем все кадры
# -----------------------------
cache = []
for t in tilda_values:
    # --- 1. Бифуркации ---
    u1_grid = np.linspace(0.01, 0.99, 500)
    den = (1 - u1_grid) * (1 - u2)
    alpha_tc = (1 - 0.5 * (1 - u2)) / den
    alpha_flip = (1 - 3*0.5*(1-u2)) / den
    alpha_ns = (3 - 2*0.5*(1-u2)) / den

    # --- 2. Фазовый портрет для нескольких начальных точек ---
    phase_trajs = []
    for x0 in x0_values:
        for y0 in y0_values:
            x_n, y_n = x0, y0
            xs, ys = [], []
            for i in range(n_iter):
                x_next = t * y_n * (1 - y_n) * (1 - u1_fixed)
                y_next = (x_n + v*y_n)*(1-u2)
                x_n, y_n = x_next, y_next
                if i >= burn_in:
                    xs.append(x_n)
                    ys.append(y_n)
            phase_trajs.append((xs, ys))

    cache.append({
        "tilda_alpha": t,
        "u1_grid": u1_grid,
        "alpha_tc": alpha_tc,
        "alpha_flip": alpha_flip,
        "alpha_ns": alpha_ns,
        "phase_trajs": phase_trajs
    })

# -----------------------------
# Создаём фигуру с 2 под графиками
# -----------------------------
fig = make_subplots(rows=1, cols=2, subplot_titles=("Бифуркации", "Фазовый портрет"))

# --- первый кадр ---
first = cache[0]

# Бифуркации
fig.add_trace(go.Scatter(x=first["u1_grid"], y=first["alpha_tc"], mode='lines', name='Транскритическая'), row=1, col=1)
fig.add_trace(go.Scatter(x=first["u1_grid"], y=first["alpha_flip"], mode='lines', name='Flip'), row=1, col=1)
fig.add_trace(go.Scatter(x=first["u1_grid"], y=first["alpha_ns"], mode='lines', name='Неймарка–Сакера'), row=1, col=1)
fig.add_trace(go.Scatter(x=[u1_fixed], y=[first["tilda_alpha"]], mode='markers', name='текущий параметр', marker=dict(color='red', size=12)), row=1, col=1)

# Фазовый портрет — несколько траекторий
for xs, ys in first["phase_trajs"]:
    fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines+markers',
                             line=dict(width=1, color='rgba(150,150,150,0.6)'),
                             marker=dict(size=4),
                             showlegend=False), row=1, col=2)

# -----------------------------
# Определяем фиксированные оси для фазового портрета
# -----------------------------
all_x = [x for frame in cache for traj in frame["phase_trajs"] for x in traj[0]]
all_y = [y for frame in cache for traj in frame["phase_trajs"] for y in traj[1]]
x_min, x_max = 0, max(all_x)*1.05
y_min, y_max = 0, max(all_y)*1.05

# -----------------------------
# Создаём кадры анимации
# -----------------------------
frames = []
for c in cache:
    phase_trajs = c["phase_trajs"]

    frame_data = [
        # Бифуркации
        go.Scatter(x=c["u1_grid"], y=c["alpha_tc"]),
        go.Scatter(x=c["u1_grid"], y=c["alpha_flip"]),
        go.Scatter(x=c["u1_grid"], y=c["alpha_ns"]),
        go.Scatter(x=[u1_fixed], y=[c["tilda_alpha"]])
    ]

    # Фазовый портрет — все траектории
    for xs, ys in phase_trajs:
        frame_data.append(go.Scatter(x=xs, y=ys, mode='lines+markers',
                                     line=dict(width=1, color='rgba(150,150,150,0.6)'),
                                     marker=dict(size=4),
                                     showlegend=False))

    frames.append(go.Frame(data=frame_data, name=str(c["tilda_alpha"])))

fig.frames = frames

# -----------------------------
# Настройки осей и легенды
# -----------------------------
fig.update_xaxes(title_text="u₁", row=1, col=1)
fig.update_yaxes(title_text="~α", row=1, col=1)

fig.update_xaxes(title_text="x", row=1, col=2, range=[x_min, x_max])
fig.update_yaxes(title_text="y", row=1, col=2, range=[y_min, y_max])

fig.update_layout(
    legend=dict(x=0, y=1, bgcolor='rgba(0,0,0,0)'),
    width=1400,
    height=500,
    title="Двойная анимация: бифуркации и фазовый портрет",
    updatemenus=[dict(type="buttons", showactive=False,
                      buttons=[dict(label="Play", method="animate",
                                    args=[None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True}]),
                               dict(label="Pause", method="animate",
                                    args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}])])]
)

fig.show()
